## Web Search Tool Calling

In [2]:
import os

from mistralai import Mistral
api_key = os.environ["MISTRAL_API_KEY"]
model = "mistral-medium-latest"

client = Mistral(api_key=api_key)


# Create a web search agent
websearch_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    description="Agent able to search information over the web.",
    name="Websearch Agent",
    instructions="You have the ability to perform web searches with `web_search` to find up-to-date information.",
    tools=[{"type": "web_search"}],
    completion_args={
        "temperature": 0.3,
        "top_p": 0.95,
    },
)
websearch_agent

Agent(model='mistral-medium-latest', name='Websearch Agent', id='ag_019cc2ce55ee7729a2b84b85d0fd0fe6', version=0, versions=[], created_at=datetime.datetime(2026, 3, 6, 11, 0, 28, 300987, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 3, 6, 11, 0, 28, 300990, tzinfo=TzInfo(0)), deployment_chat=False, source='api', instructions='You have the ability to perform web searches with `web_search` to find up-to-date information.', tools=[WebSearchTool(type='web_search')], completion_args=CompletionArgs(stop=None, presence_penalty=None, frequency_penalty=None, temperature=0.3, top_p=0.95, max_tokens=None, random_seed=None, prediction=None, response_format=None, tool_choice='auto'), description='Agent able to search information over the web.', handoffs=None, metadata=None, object='agent', version_message=None)

In [3]:
# Start a conversation with a web search query
response = client.beta.conversations.start(
    agent_id=websearch_agent.id,
    inputs="What are the latest news about Münster, Germany?",
)
response

ConversationResponse(conversation_id='conv_019cc2ce569072af819e5b54a0f6a7ae', outputs=[ToolExecutionEntry(name='web_search', arguments='{"query": "latest news Münster Germany March 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 3, 6, 11, 0, 29, 131714, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 3, 6, 11, 0, 30, 71391, tzinfo=TzInfo(0)), id='tool_exec_019cc2ce594b77e39f800efbdc7341a0', info={}), MessageOutputEntry(content=[TextChunk(text='Here are some of the latest news and updates related to Münster, Germany, as of March 2026:\n\n1. **German Conference on Synthetic Biology (GCSB 2026)**: Münster is hosting the German Conference on Synthetic Biology from March 11–13, 2026. This event is expected to bring together researchers and industry professionals from across the country and beyond, focusing on the latest advancements in synthetic biology', type='text'), ToolReferenceChunk(tool='web_search', title='GCSB 2026', type='tool_reference', 

In [13]:
# Build a markdown string with inline citations
from IPython.display import Markdown

output = response.outputs[-1]
parts = []
for chunk in output.content:
    if chunk.type == "text":
        parts.append(chunk.text)
    elif chunk.type == "tool_reference":
        parts.append(f"[{chunk.title}]({chunk.url})")

Markdown("".join(parts))

Here are some of the latest news and updates related to Münster, Germany, as of March 2026:

1. **German Conference on Synthetic Biology (GCSB 2026)**: Münster is hosting the German Conference on Synthetic Biology from March 11–13, 2026. This event is expected to bring together researchers and industry professionals from across the country and beyond, focusing on the latest advancements in synthetic biology[GCSB 2026](https://gcsb2026.de/).

2. **Münster/Osnabrück Airport (FMO)**: The airport remains one of the few in Germany operating 24/7, continuing to play a key role in regional and international air transport. Recent reports highlight its ongoing importance for route development and connectivity[Airport In Focus: Münster/Osnabrück Airport | Aviation Week Network](https://aviationweek.com/air-transport/airports-networks/airport-focus-munsterosnabruck-airport).

3. **National and Regional Developments**: While not specific to Münster, broader national changes in March 2026 include the extension of temporary protection for Ukrainian refugees until March 2027, adjustments to the German pension system, and increased transparency in the SCHUFA credit scoring system. These changes may impact residents and expats in Münster as well[Everything that changes in Germany in March 2026](https://www.thelocal.de/20260219/everything-that-changes-in-germany-in-march-2026)[March 2026: 10 changes affecting expats in Germany](https://www.iamexpat.de/expat-info/germany-news/march-2026-10-changes-affecting-expats-germany).

If you are interested in more localized news, such as cultural events or city-specific updates, let me know and I can look for more details!

In [12]:
response

ConversationResponse(conversation_id='conv_019cc2ce569072af819e5b54a0f6a7ae', outputs=[ToolExecutionEntry(name='web_search', arguments='{"query": "latest news Münster Germany March 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 3, 6, 11, 0, 29, 131714, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 3, 6, 11, 0, 30, 71391, tzinfo=TzInfo(0)), id='tool_exec_019cc2ce594b77e39f800efbdc7341a0', info={}), MessageOutputEntry(content=[TextChunk(text='Here are some of the latest news and updates related to Münster, Germany, as of March 2026:\n\n1. **German Conference on Synthetic Biology (GCSB 2026)**: Münster is hosting the German Conference on Synthetic Biology from March 11–13, 2026. This event is expected to bring together researchers and industry professionals from across the country and beyond, focusing on the latest advancements in synthetic biology', type='text'), ToolReferenceChunk(tool='web_search', title='GCSB 2026', type='tool_reference', 

## Test: Restrict search to specific sites

The web_search tool has no built-in site filter parameter. Let's test if we can steer it via:
1. Using `site:` operator in the user query
2. Instructing the agent to only search specific domains

In [16]:
# Test 1: site: operator in the user query
response_site = client.beta.conversations.start(
    agent_id=websearch_agent.id,
    inputs="site:muenster.de Was gibt es Neues in Münster?",
)

# Show the search query the model actually used
for entry in response_site.outputs:
    if entry.type == "tool.execution":
        print(f"Search query used: {entry.arguments}")

# Render the response
output_site = response_site.outputs[-1]
parts = []
for chunk in output_site.content:
    if chunk.type == "text":
        parts.append(chunk.text)
    elif chunk.type == "tool_reference":
        parts.append(f" [{chunk.title}]({chunk.url})")
Markdown("".join(parts))

Search query used: {"query": "site:muenster.de Neuigkeiten Aktuelles 2026"}


In Münster gibt es 2026 einige spannende Neuigkeiten und Veränderungen:

- Ab 2026 entstehen im Baugebiet „Am Dornbusch“ in Amelsbüren rund 200 neue Wohneinheiten und eine Kindertagesstätte. Die Stadt informiert auf ihrer Webseite über die Vermarktung der Baugrundstücke [Muenster](https://www.muenster.de/pressemeldungen/web/frontend/output/immobilien_param_year/search/1/page/1/show/1152117/design/standard).
- Die Parkgebühren in Münster steigen zum 1. Januar 2026 deutlich an, besonders in der zentralen Parkzone I, wo das Parken dann 3,50 Euro pro angefangene Stunde kosten soll [Parken in Münster soll 2026 teurer werden](https://www.muenster.de/pressemeldungen/web/frontend/design/kommunikation/show/1204205).
- Die Schmutzwassergebühr wird 2026 um 24 Cent pro Quadratmeter auf 3,11 Euro erhöht, während die Niederschlagswassergebühr unverändert bleibt [Stadt Münster: Amt für Kommunikation - Pressemeldungen](https://www.muenster.de/pressemeldungen/web/frontend/design/kommunikation/show/1204955).
- Der Radrenn-Klassiker startet 2026 erstmals in Borken und führt am Tag der Deutschen Einheit (3. Oktober) durch den Kreis Borken und Coesfeld nach Münster [Stadt Münster: Münster Marketing - Veranstaltungskalender](https://www.muenster.de/veranstaltungskalender/scripts/frontend/tourismus/top-veranstaltungen.php?guestID=101).
- Das Stadtfest „Münster Mittendrin“ findet 2026 am letzten August-Wochenende statt [Stadtfest „Münster Mittendrin“ wird 2026 Teil des Nordrhein ...](https://www.muenster.de/pressemeldungen/web/frontend/design/kommunikation/show/1202085).
- Die Stadt Münster veröffentlicht 2026 eine Fortschreibung ihres Berichts zum kommunalen Klima-Controlling, mit neuen Fahrradstraßen, zusätzlichen Photovoltaikanlagen und mehr grüner Wärme [Stadt Münster: Tiefbauamt - Pressemeldungen](https://www.muenster.de/pressemeldungen/web/frontend/output/standard/search/1/design/tiefbauamt/page/1/show/1206329).

Möchtest du zu einem dieser Themen mehr Details wissen?

In [15]:
# Test 2: Agent with instructions to only search specific domains
site_restricted_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Site-Restricted Agent",
    description="Agent that searches only on muenster.de",
    instructions=(
        "You have the ability to perform web searches with `web_search`. "
        "IMPORTANT: Always include 'site:muenster.de' in your search queries. "
        "Only return information from muenster.de."
    ),
    tools=[{"type": "web_search"}],
    completion_args={"temperature": 0.3, "top_p": 0.95},
)

response_restricted = client.beta.conversations.start(
    agent_id=site_restricted_agent.id,
    inputs="Was gibt es Neues in Münster?",
)

# Show the search query the model actually used
for entry in response_restricted.outputs:
    if entry.type == "tool.execution":
        print(f"Search query used: {entry.arguments}")

# Render the response
output_restricted = response_restricted.outputs[-1]
parts = []
for chunk in output_restricted.content:
    if chunk.type == "text":
        parts.append(chunk.text)
    elif chunk.type == "tool_reference":
        parts.append(f" [{chunk.title}]({chunk.url})")
Markdown("".join(parts))

Search query used: {"query": "Aktuelle Nachrichten Münster site:muenster.de"}


In Münster gibt es aktuell einige spannende Entwicklungen und Termine:

- Die Stadt sucht neue Namen für den Lüderitz- und den Woermannweg in Gremmendorf. Bürgerinnen und Bürger können bis zum 22. März Vorschläge einreichen.
- Vom 28. bis 30. August 2025 findet der Nordrhein-Westfalen-Tag in Münster statt, bei dem der 80. Geburtstag des Landes mit verschiedenen Veranstaltungen und zwölf Bühnen in der Innenstadt gefeiert wird.
- Oberbürgermeister Tilman Fuchs tauschte sich kürzlich mit Vertretern aus Wirtschaft und Hochschulen über die kommunale Wärmeplanung und Klimaarbeit im Stadtkonzern aus.
- Die Weihnachtsmärkte in Münster finden unter erhöhter Wachsamkeit statt, nachdem es in Magdeburg einen Anschlag gab. Die Stadt zeigt sich solidarisiert mit den Betroffenen.

Für weitere Details und aktuelle Veranstaltungen kannst du den Veranstaltungskalender der Stadt Münster besuchen [Startseite | Stadt Münster](https://www.muenster.de/) [web.muenster.de - Münster in Westfalen: Veranstaltungskalender](https://web.muenster.de/veranstaltungskalender.html). Möchtest du zu einem bestimmten Thema mehr wissen?